# ETL OpenFoodFacts - Datamart Nutrition & Qualite v7

**Module** : TRDE703 - Atelier Integration des Donnees (M1)  
**Technologie** : Apache Spark (PySpark) -> PostgreSQL  
**Architecture** : Bronze -> Silver -> Gold -> Datamart

---

## Sommaire
1. Configuration et imports
2. **BRONZE** - Ingestion des donnees brutes
3. **METRIQUES BEFORE** - Qualite avant nettoyage
4. **SILVER** - Nettoyage et conformite
5. **METRIQUES AFTER** - Qualite apres nettoyage
6. **GOLD** - Modelisation dimensionnelle
7. Chargement PostgreSQL
8. **COMPARATIF BEFORE/AFTER** - Cahier de qualite

In [ ]:
"""
Script de creation de la base de donnees PostgreSQL pour OpenFoodFacts
Executer ce script UNE SEULE FOIS avant de lancer le notebook ETL
"""

import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# ============================================
# CONFIGURATION
# ============================================
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"  # Modifier ici
DB_NAME = "openfoodfacts_dw"

# ============================================
# ETAPE 1 : Creer la base de donnees
# ============================================
def create_database():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD
        )
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cursor = conn.cursor()

        cursor.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
        exists = cursor.fetchone()

        if exists:
            cursor.execute(f"""
                SELECT pg_terminate_backend(pg_stat_activity.pid)
                FROM pg_stat_activity
                WHERE pg_stat_activity.datname = '{DB_NAME}'
                AND pid <> pg_backend_pid();
            """)
            cursor.execute(f"DROP DATABASE {DB_NAME}")
            print(f"Base '{DB_NAME}' supprimee")

        cursor.execute(f"CREATE DATABASE {DB_NAME} WITH ENCODING = 'UTF8' TEMPLATE = template0;")
        print(f"Base '{DB_NAME}' creee (UTF-8)")

        cursor.close()
        conn.close()
        return True

    except Exception as e:
        print(f"Erreur creation base: {e}")
        return False

# ============================================
# ETAPE 2 : Creer les tables avec FK et index
# ============================================
def create_tables():
    ddl = """
    -- Supprimer les tables existantes
    DROP TABLE IF EXISTS fact_nutrition_snapshot CASCADE;
    DROP TABLE IF EXISTS bridge_product_category CASCADE;
    DROP TABLE IF EXISTS dim_product CASCADE;
    DROP TABLE IF EXISTS dim_nutri CASCADE;
    DROP TABLE IF EXISTS dim_category CASCADE;
    DROP TABLE IF EXISTS dim_country CASCADE;
    DROP TABLE IF EXISTS dim_brand CASCADE;
    DROP TABLE IF EXISTS dim_time CASCADE;

    -- dim_time
    CREATE TABLE dim_time (
        time_sk BIGINT PRIMARY KEY,
        date DATE NOT NULL,
        year INTEGER NOT NULL,
        month INTEGER NOT NULL,
        day INTEGER NOT NULL,
        week INTEGER NOT NULL,
        iso_week INTEGER NOT NULL
    );

    -- dim_brand
    CREATE TABLE dim_brand (
        brand_sk BIGINT PRIMARY KEY,
        brand_name VARCHAR(500) NOT NULL
    );

    -- dim_country
    CREATE TABLE dim_country (
        country_sk BIGINT PRIMARY KEY,
        country_code VARCHAR(100) NOT NULL,
        country_name_fr VARCHAR(255)
    );

    -- dim_category
    CREATE TABLE dim_category (
        category_sk BIGINT PRIMARY KEY,
        category_code VARCHAR(500) NOT NULL,
        category_name_fr VARCHAR(500),
        level INTEGER,
        parent_category_sk BIGINT REFERENCES dim_category(category_sk)
    );

    -- dim_nutri
    CREATE TABLE dim_nutri (
        nutri_sk BIGINT PRIMARY KEY,
        nutriscore_grade VARCHAR(1),
        nova_group INTEGER,
        ecoscore_grade VARCHAR(1)
    );

    -- dim_product avec FK
    CREATE TABLE dim_product (
        product_sk BIGINT PRIMARY KEY,
        code VARCHAR(50) NOT NULL,
        product_name VARCHAR(1000),
        brand_sk BIGINT REFERENCES dim_brand(brand_sk),
        primary_category_sk BIGINT REFERENCES dim_category(category_sk),
        countries_multi TEXT,
        effective_from DATE,
        effective_to DATE,
        is_current BOOLEAN DEFAULT TRUE
    );

    -- bridge_product_category avec FK
    CREATE TABLE bridge_product_category (
        product_sk BIGINT REFERENCES dim_product(product_sk),
        category_sk BIGINT REFERENCES dim_category(category_sk),
        PRIMARY KEY (product_sk, category_sk)
    );

    -- fact_nutrition_snapshot avec FK
    CREATE TABLE fact_nutrition_snapshot (
        fact_id BIGINT PRIMARY KEY,
        product_sk BIGINT REFERENCES dim_product(product_sk),
        time_sk BIGINT REFERENCES dim_time(time_sk),
        energy_kcal_100g DECIMAL(10,2),
        fat_100g DECIMAL(10,2),
        saturated_fat_100g DECIMAL(10,2),
        sugars_100g DECIMAL(10,2),
        salt_100g DECIMAL(10,2),
        proteins_100g DECIMAL(10,2),
        fiber_100g DECIMAL(10,2),
        sodium_100g DECIMAL(10,2),
        nutriscore_grade VARCHAR(1),
        nova_group INTEGER,
        ecoscore_grade VARCHAR(1),
        completeness_score DECIMAL(5,4),
        quality_issues_json TEXT
    );

    -- Index pour performances
    CREATE INDEX idx_fact_product ON fact_nutrition_snapshot(product_sk);
    CREATE INDEX idx_fact_time ON fact_nutrition_snapshot(time_sk);
    CREATE INDEX idx_fact_nutriscore ON fact_nutrition_snapshot(nutriscore_grade);
    CREATE INDEX idx_product_brand ON dim_product(brand_sk);
    CREATE INDEX idx_product_category ON dim_product(primary_category_sk);
    CREATE INDEX idx_product_code ON dim_product(code);
    CREATE INDEX idx_category_level ON dim_category(level);
    CREATE INDEX idx_time_date ON dim_time(date);
    """

    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        cursor = conn.cursor()
        cursor.execute(ddl)
        conn.commit()
        cursor.close()
        conn.close()

        print("Tables creees:")
        print("  dim_time, dim_brand, dim_country, dim_category")
        print("  dim_nutri, dim_product, bridge_product_category")
        print("  fact_nutrition_snapshot")
        print("FK et index configures")
        return True

    except Exception as e:
        print(f"Erreur: {e}")
        return False

# ============================================
# ETAPE 3 : Verifier
# ============================================
def verify():
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        cursor = conn.cursor()

        cursor.execute("""
            SELECT table_name FROM information_schema.tables
            WHERE table_schema = 'public' ORDER BY table_name;
        """)
        tables = cursor.fetchall()

        print(f"\n{len(tables)} tables creees:")
        for t in tables:
            print(f"  - {t[0]}")

        cursor.close()
        conn.close()
        return True

    except Exception as e:
        print(f"Erreur: {e}")
        return False

# ============================================
# EXECUTION
# ============================================
if __name__ == "__main__":
    print("=" * 40)
    print("Setup PostgreSQL - OpenFoodFacts")
    print("=" * 40)
    
    if create_database():
        if create_tables():
            verify()
    
    print("\n" + "=" * 40)
    print("Pret! Lancez le notebook ETL")
    print("=" * 40)


---
## 1. Configuration et Imports

In [1]:
# CELLULE 1 - Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType
from pyspark.sql.functions import year, month, dayofmonth, weekofyear
from datetime import datetime
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
import json

print("Imports OK")

Imports OK


In [2]:
# CELLULE 2 - Session Spark
spark = SparkSession.builder \
    .appName("OpenFoodFacts_ETL_v7") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} initialise")

Spark 4.1.1 initialise


In [3]:
# CELLULE 3 - Configuration
INPUT_PATH = "data/bronze/en.openfoodfacts.org.products_echantillon_10000.csv"

PG_HOST = "localhost"
PG_PORT = "5432"
PG_DATABASE = "openfoodfacts_dw"
PG_USER = "postgres"
PG_PASSWORD = "postgres"

PG_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}?charSet=UTF-8"
PG_PROPS = {"user": PG_USER, "password": PG_PASSWORD, "driver": "org.postgresql.Driver"}

# Regles de qualite
NUTRIENT_BOUNDS = {
    "energy_kcal_100g": (0, 900),
    "fat_100g": (0, 100),
    "saturated_fat_100g": (0, 100),
    "sugars_100g": (0, 100),
    "salt_100g": (0, 100),
    "proteins_100g": (0, 100),
    "fiber_100g": (0, 100),
    "sodium_100g": (0, 40)
}

SALT_SODIUM_FACTOR = 2.5
MIN_NUTRIENTS = 3

# Colonnes nutriments pour statistiques
nutrient_cols = ["energy_kcal_100g", "fat_100g", "saturated_fat_100g", 
                 "sugars_100g", "salt_100g", "proteins_100g", "fiber_100g"]

print(f"Source: {INPUT_PATH}")
print(f"Cible: PostgreSQL {PG_DATABASE}")

Source: data/bronze/en.openfoodfacts.org.products_echantillon_10000.csv
Cible: PostgreSQL openfoodfacts_dw


In [4]:
# CELLULE 4 - Fonctions utilitaires

def detect_separator(path):
    with open(path, 'r', encoding='utf-8') as f:
        line = f.readline()
    if line.count('\t') > max(line.count(';'), line.count(',')):
        return '\t'
    if line.count(';') > line.count(','):
        return ';'
    return ','

def clean_numeric(col_name):
    c = F.regexp_replace(F.col(col_name), ",", ".")
    c = F.when(c.like("http%"), None).otherwise(c)
    c = F.when(c.rlike(r'^[+-]?[0-9]+(\.[0-9]+)?([eE][+-]?[0-9]+)?$'), c.cast("double")).otherwise(None)
    return c

def load_to_pg(df, table_name):
    try:
        conn = psycopg2.connect(host=PG_HOST, port=int(PG_PORT), 
                                database=PG_DATABASE, user=PG_USER, password=PG_PASSWORD)
        conn.set_client_encoding('UTF8')
        cur = conn.cursor()
        cur.execute(f"TRUNCATE TABLE {table_name} CASCADE")
        conn.commit()
        cur.close()
        conn.close()
        df.write.jdbc(url=PG_URL, table=table_name, mode="append", properties=PG_PROPS)
        count = df.count()
        print(f"   {table_name}: {count} lignes")
        return count
    except Exception as e:
        print(f"   ERREUR {table_name}: {e}")
        return 0

print("Fonctions definies")

Fonctions definies


---
## 2. BRONZE - Ingestion des donnees brutes

In [5]:
# CELLULE 5 - Lecture Bronze
sep = detect_separator(INPUT_PATH)
print(f"Separateur detecte: '{sep}'")

df_raw = spark.read \
    .option("header", "true") \
    .option("sep", sep) \
    .option("inferSchema", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv(INPUT_PATH)

bronze_count = df_raw.count()
print(f"BRONZE: {bronze_count} lignes, {len(df_raw.columns)} colonnes")

Separateur detecte: ';'
BRONZE: 10000 lignes, 27 colonnes


In [6]:
# CELLULE 6 - Selection des colonnes avec typage explicite
df = df_raw.select(
    F.col("code").cast("string"),
    F.col("product_name").cast("string"),
    F.col("brands").cast("string"),
    F.col("categories_tags").cast("string"),
    F.col("countries_tags").cast("string"),
    F.col("nutriscore_grade").cast("string"),
    clean_numeric("nova_group").cast("int").alias("nova_group"),
    F.col("environmental_score_grade").cast("string").alias("ecoscore_grade"),
    clean_numeric("energy-kcal_100g").alias("energy_kcal_100g"),
    clean_numeric("fat_100g").alias("fat_100g"),
    clean_numeric("saturated-fat_100g").alias("saturated_fat_100g"),
    clean_numeric("sugars_100g").alias("sugars_100g"),
    clean_numeric("salt_100g").alias("salt_100g"),
    clean_numeric("proteins_100g").alias("proteins_100g"),
    clean_numeric("fiber_100g").alias("fiber_100g"),
    clean_numeric("sodium_100g").alias("sodium_100g"),
    clean_numeric("completeness").alias("completeness"),
    F.col("data_quality_errors_tags").cast("string").alias("quality_errors"),
    F.col("last_modified_datetime").cast("string").alias("last_modified_datetime"),
    clean_numeric("last_modified_t").cast("long").alias("last_modified_t")
)

print(f"Colonnes selectionnees: {len(df.columns)}")
print(f"Colonnes: {df.columns}")

Colonnes selectionnees: 20
Colonnes: ['code', 'product_name', 'brands', 'categories_tags', 'countries_tags', 'nutriscore_grade', 'nova_group', 'ecoscore_grade', 'energy_kcal_100g', 'fat_100g', 'saturated_fat_100g', 'sugars_100g', 'salt_100g', 'proteins_100g', 'fiber_100g', 'sodium_100g', 'completeness', 'quality_errors', 'last_modified_datetime', 'last_modified_t']


---
## 3. METRIQUES BEFORE - Qualite avant nettoyage

Calcul des indicateurs de qualite sur les donnees **BRUTES** avant application des regles de nettoyage.

In [7]:
# CELLULE 7 - Metriques BEFORE (avant nettoyage)
print("=" * 65)
print("  METRIQUES BEFORE - Donnees brutes (avant nettoyage)")
print("=" * 65)

before_count = df.count()
print(f"\nTotal lignes: {before_count}")

# Dictionnaires pour stocker les metriques BEFORE
completude_before = {}
anomalies_before = {}

print(f"\n--- Completude par champ (BEFORE) ---")

# Colonnes texte
cols_texte = ["product_name", "brands", "categories_tags", "nutriscore_grade", "ecoscore_grade"]
for c in cols_texte:
    non_null = df.filter(F.col(c).isNotNull() & (F.col(c) != "")).count()
    pct = round(non_null / before_count * 100, 1)
    completude_before[c] = pct
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    print(f"  {c:25}: [{bar}] {pct:5.1f}%")

# Colonnes numeriques
cols_num = ["nova_group"] + nutrient_cols
for c in cols_num:
    non_null = df.filter(F.col(c).isNotNull()).count()
    pct = round(non_null / before_count * 100, 1)
    completude_before[c] = pct
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    print(f"  {c:25}: [{bar}] {pct:5.1f}%")

# Moyennes
nutrient_pcts = [completude_before.get(c, 0) for c in nutrient_cols]
avg_nutrient_before = round(sum(nutrient_pcts) / len(nutrient_pcts), 1)
avg_global_before = round(sum(completude_before.values()) / len(completude_before), 1)

print(f"\n--- Moyennes BEFORE ---")
print(f"  Completude nutriments: {avg_nutrient_before}%")
print(f"  Completude globale:    {avg_global_before}%")

  METRIQUES BEFORE - Donnees brutes (avant nettoyage)

Total lignes: 10000

--- Completude par champ (BEFORE) ---
  product_name             : [##################..]  92.4%
  brands                   : [############........]  61.4%
  categories_tags          : [########............]  42.7%
  nutriscore_grade         : [###################.]  98.9%
  ecoscore_grade           : [#############.......]  69.6%
  nova_group               : [#####...............]  25.7%
  energy_kcal_100g         : [##############......]  71.6%
  fat_100g                 : [##############......]  72.1%
  saturated_fat_100g       : [#############.......]  67.2%
  sugars_100g              : [#############.......]  68.5%
  salt_100g                : [############........]  62.5%
  proteins_100g            : [##############......]  72.3%
  fiber_100g               : [#######.............]  35.9%

--- Moyennes BEFORE ---
  Completude nutriments: 64.3%
  Completude globale:    64.7%


In [8]:
# CELLULE 8 - Detection des anomalies BEFORE
print("\n--- Anomalies detectees (BEFORE) ---")

# Codes vides
codes_vides = df.filter(F.col("code").isNull() | (F.col("code") == "")).count()
anomalies_before["codes_vides"] = codes_vides
print(f"  Codes vides/nuls:        {codes_vides}")

# Noms vides
noms_vides = df.filter(F.col("product_name").isNull() | (F.col("product_name") == "")).count()
anomalies_before["noms_vides"] = noms_vides
print(f"  Noms produits vides:     {noms_vides}")

# Doublons
total_codes = df.filter(F.col("code").isNotNull() & (F.col("code") != "")).count()
codes_uniques = df.filter(F.col("code").isNotNull() & (F.col("code") != "")).select("code").distinct().count()
doublons = total_codes - codes_uniques
anomalies_before["doublons_code"] = doublons
print(f"  Doublons code-barres:    {doublons}")

# Valeurs hors bornes
hors_bornes_total = 0
print(f"  Valeurs hors bornes:")
for col, (min_v, max_v) in NUTRIENT_BOUNDS.items():
    if col in df.columns:
        count_hb = df.filter((F.col(col) < min_v) | (F.col(col) > max_v)).count()
        if count_hb > 0:
            print(f"    - {col}: {count_hb} hors [{min_v}, {max_v}]")
            hors_bornes_total += count_hb
anomalies_before["hors_bornes"] = hors_bornes_total
if hors_bornes_total == 0:
    print(f"    (aucune)")

# Incoherence saturated > fat
incoherence_fat = df.filter(
    F.col("saturated_fat_100g").isNotNull() & 
    F.col("fat_100g").isNotNull() & 
    (F.col("saturated_fat_100g") > F.col("fat_100g"))
).count()
anomalies_before["incoherence_fat"] = incoherence_fat
print(f"  Incoherence sat_fat>fat: {incoherence_fat}")

total_anomalies = sum(anomalies_before.values())
print(f"\n  >>> TOTAL ANOMALIES: {total_anomalies}")


--- Anomalies detectees (BEFORE) ---
  Codes vides/nuls:        0
  Noms produits vides:     763
  Doublons code-barres:    2799
  Valeurs hors bornes:
    - energy_kcal_100g: 40 hors [0, 900]
    - fat_100g: 13 hors [0, 100]
    - saturated_fat_100g: 6 hors [0, 100]
    - sugars_100g: 17 hors [0, 100]
    - salt_100g: 3 hors [0, 100]
    - proteins_100g: 8 hors [0, 100]
    - fiber_100g: 3 hors [0, 100]
    - sodium_100g: 3 hors [0, 40]
  Incoherence sat_fat>fat: 11

  >>> TOTAL ANOMALIES: 3666


---
## 4. SILVER - Nettoyage et Conformite

**Regles de qualite appliquees** :
1. Filtrage : code et product_name obligatoires
2. Deduplication : par code-barres, garder le plus recent (last_modified_t)
3. Bornes : nutriments dans intervalles valides
4. Coherence : saturated_fat <= fat
5. Harmonisation : conversion sel/sodium
6. Completude : minimum 3 nutriments renseignes

In [9]:
# CELLULE 9 - Filtrage et deduplication
print("=== NETTOYAGE ===")

# 9.1 Filtrer : code et nom obligatoires
df = df.filter(F.col("code").isNotNull() & (F.col("code") != ""))
df = df.filter(F.col("product_name").isNotNull() & (F.col("product_name") != ""))
count_after_filter = df.count()
print(f"1. Filtrage code/nom: {before_count} -> {count_after_filter} (-{before_count - count_after_filter})")

# 9.2 Deduplication : garder le plus recent
window_dedup = Window.partitionBy("code").orderBy(F.col("last_modified_t").desc_nulls_last())
df = df.withColumn("_rank", F.row_number().over(window_dedup))
df = df.filter(F.col("_rank") == 1).drop("_rank")
count_after_dedup = df.count()
print(f"2. Deduplication:     {count_after_filter} -> {count_after_dedup} (-{count_after_filter - count_after_dedup})")

=== NETTOYAGE ===
1. Filtrage code/nom: 10000 -> 9237 (-763)
2. Deduplication:     9237 -> 6695 (-2542)


In [10]:
# CELLULE 10 - Regles de qualite

# 10.1 Appliquer bornes nutritionnelles (mettre NULL si hors bornes)
for col_name, (min_v, max_v) in NUTRIENT_BOUNDS.items():
    if col_name in df.columns:
        df = df.withColumn(col_name, 
            F.when((F.col(col_name) >= min_v) & (F.col(col_name) <= max_v), F.col(col_name))
            .otherwise(None))
print("3. Bornes nutritionnelles appliquees")

# 10.2 Coherence : saturated_fat <= fat
df = df.withColumn("saturated_fat_100g", 
    F.when(F.col("saturated_fat_100g") > F.col("fat_100g"), None)
    .otherwise(F.col("saturated_fat_100g")))
print("4. Coherence saturated_fat <= fat")

# 10.3 Harmonisation sel/sodium
df = df.withColumn("sodium_100g", 
    F.when(F.col("sodium_100g").isNull() & F.col("salt_100g").isNotNull(),
           F.round(F.col("salt_100g") / SALT_SODIUM_FACTOR, 2))
    .otherwise(F.col("sodium_100g")))
df = df.withColumn("salt_100g", 
    F.when(F.col("salt_100g").isNull() & F.col("sodium_100g").isNotNull(),
           F.round(F.col("sodium_100g") * SALT_SODIUM_FACTOR, 2))
    .otherwise(F.col("salt_100g")))
print("5. Harmonisation sel/sodium (facteur 2.5)")

3. Bornes nutritionnelles appliquees
4. Coherence saturated_fat <= fat
5. Harmonisation sel/sodium (facteur 2.5)


In [11]:
# CELLULE 11 - Filtres de completude

# 11.1 Compter nutriments disponibles
nutrient_count_expr = F.lit(0)
for c in nutrient_cols:
    nutrient_count_expr = nutrient_count_expr + F.when(F.col(c).isNotNull(), 1).otherwise(0)
df = df.withColumn("_n_count", nutrient_count_expr)

# 11.2 Filtrer : minimum 3 nutriments
df = df.filter(F.col("_n_count") >= MIN_NUTRIENTS)
count_after_nutrients = df.count()
print(f"6. Min {MIN_NUTRIENTS} nutriments: {count_after_dedup} -> {count_after_nutrients} (-{count_after_dedup - count_after_nutrients})")

# 11.3 Somme nutriments <= 110g
df = df.withColumn("_sum", 
    F.coalesce(F.col("fat_100g"), F.lit(0)) +
    F.coalesce(F.col("sugars_100g"), F.lit(0)) +
    F.coalesce(F.col("proteins_100g"), F.lit(0)) +
    F.coalesce(F.col("fiber_100g"), F.lit(0)) +
    F.coalesce(F.col("salt_100g"), F.lit(0)))
df = df.filter(F.col("_sum") <= 110)
count_after_sum = df.count()
print(f"7. Somme <= 110g:     {count_after_nutrients} -> {count_after_sum} (-{count_after_nutrients - count_after_sum})")

6. Min 3 nutriments: 6695 -> 5127 (-1568)
7. Somme <= 110g:     5127 -> 5112 (-15)


In [12]:
# CELLULE 12 - Normalisation

# 12.1 Normaliser scores
df = df.withColumn("nutriscore_grade", 
    F.when(F.upper(F.col("nutriscore_grade")).isin(["A","B","C","D","E"]), 
           F.upper(F.col("nutriscore_grade"))).otherwise(None))
df = df.withColumn("nova_group", 
    F.when(F.col("nova_group").between(1, 4), F.col("nova_group")).otherwise(None))
df = df.withColumn("ecoscore_grade", 
    F.when(F.upper(F.col("ecoscore_grade")).isin(["A","B","C","D","E"]), 
           F.upper(F.col("ecoscore_grade"))).otherwise(None))
print("8. Scores normalises")

# 12.2 Extraire valeurs principales
df = df.withColumn("brand_primary", F.trim(F.split(F.col("brands"), ",").getItem(0)))
df = df.withColumn("primary_category_code", F.trim(F.split(F.col("categories_tags"), ",").getItem(0)))
df = df.withColumn("primary_country_code", F.trim(F.split(F.col("countries_tags"), ",").getItem(0)))

# 12.3 JSON multi-valeurs
df = df.withColumn("countries_multi",
    F.when(F.col("countries_tags").isNotNull() & (F.col("countries_tags") != ""),
           F.concat(F.lit('["'), F.regexp_replace(F.col("countries_tags"), ",", '","'), F.lit('"]')))
    .otherwise(F.lit('[]')))

df = df.withColumn("quality_issues_json",
    F.when(F.col("quality_errors").isNotNull() & (F.col("quality_errors") != ""),
           F.concat(F.lit('["'), F.regexp_replace(F.col("quality_errors"), ",", '","'), F.lit('"]')))
    .otherwise(F.lit('[]')))

print("9. Valeurs principales et JSON crees")

8. Scores normalises
9. Valeurs principales et JSON crees


In [13]:
# CELLULE 13 - Finalisation Silver

# 13.1 Completeness score (0-1)
df = df.withColumn("completeness_score",
    F.when(F.col("completeness").isNotNull(), F.round(F.col("completeness"), 4))
    .otherwise(F.round(
        (F.when(F.col("product_name").isNotNull(), 1).otherwise(0) +
         F.when(F.col("brands").isNotNull(), 1).otherwise(0) +
         F.when(F.col("categories_tags").isNotNull(), 1).otherwise(0) +
         (F.col("_n_count") / len(nutrient_cols))) / 4.0, 4)))

# 13.2 Arrondir mesures
for c in ["energy_kcal_100g", "fat_100g", "saturated_fat_100g", "sugars_100g", 
          "salt_100g", "proteins_100g", "fiber_100g", "sodium_100g"]:
    df = df.withColumn(c, F.round(F.col(c), 2))

# 13.3 Nettoyer "unknown"
for c in ["brand_primary", "primary_category_code", "primary_country_code"]:
    df = df.withColumn(c, F.when(F.lower(F.col(c)) == "unknown", None).otherwise(F.col(c)))

# 13.4 Supprimer colonnes temporaires
df_silver = df.drop("_n_count", "_sum", "completeness", "quality_errors")

silver_count = df_silver.count()
print(f"\n{'='*60}")
print(f"  SILVER FINAL: {silver_count} produits")
print(f"  Taux de retention: {silver_count/before_count*100:.1f}%")
print(f"{'='*60}")


  SILVER FINAL: 5112 produits
  Taux de retention: 51.1%


---
## 5. METRIQUES AFTER - Qualite apres nettoyage

In [14]:
# CELLULE 14 - Metriques AFTER (apres nettoyage)
print("=" * 65)
print("  METRIQUES AFTER - Donnees nettoyees")
print("=" * 65)

after_count = df_silver.count()
print(f"\nTotal lignes: {after_count}")

# Dictionnaire pour stocker les metriques AFTER
completude_after = {}

print(f"\n--- Completude par champ (AFTER) ---")

# Colonnes texte
cols_texte_after = ["product_name", "brands", "categories_tags", "nutriscore_grade", "ecoscore_grade"]
for c in cols_texte_after:
    non_null = df_silver.filter(F.col(c).isNotNull() & (F.col(c) != "")).count()
    pct = round(non_null / after_count * 100, 1)
    completude_after[c] = pct
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    print(f"  {c:25}: [{bar}] {pct:5.1f}%")

# Colonnes numeriques
cols_num_after = ["nova_group"] + nutrient_cols
for c in cols_num_after:
    non_null = df_silver.filter(F.col(c).isNotNull()).count()
    pct = round(non_null / after_count * 100, 1)
    completude_after[c] = pct
    bar = '#' * int(pct/5) + '.' * (20 - int(pct/5))
    print(f"  {c:25}: [{bar}] {pct:5.1f}%")

# Moyennes
nutrient_pcts_after = [completude_after.get(c, 0) for c in nutrient_cols]
avg_nutrient_after = round(sum(nutrient_pcts_after) / len(nutrient_pcts_after), 1)
avg_global_after = round(sum(completude_after.values()) / len(completude_after), 1)

print(f"\n--- Moyennes AFTER ---")
print(f"  Completude nutriments: {avg_nutrient_after}%")
print(f"  Completude globale:    {avg_global_after}%")

# Completude moyenne du champ completeness_score
avg_completeness_score = df_silver.agg(F.avg("completeness_score")).first()[0]
print(f"  Score completude moyen: {avg_completeness_score:.2%}")

  METRIQUES AFTER - Donnees nettoyees

Total lignes: 5112

--- Completude par champ (AFTER) ---
  product_name             : [####################] 100.0%
  brands                   : [#############.......]  65.8%
  categories_tags          : [#########...........]  46.9%
  nutriscore_grade         : [########............]  41.2%
  ecoscore_grade           : [###.................]  16.1%
  nova_group               : [######..............]  30.9%
  energy_kcal_100g         : [###################.]  98.0%
  fat_100g                 : [###################.]  99.4%
  saturated_fat_100g       : [##################..]  90.5%
  sugars_100g              : [##################..]  92.5%
  salt_100g                : [#################...]  85.1%
  proteins_100g            : [###################.]  99.7%
  fiber_100g               : [##########..........]  51.1%

--- Moyennes AFTER ---
  Completude nutriments: 88.0%
  Completude globale:    70.6%
  Score completude moyen: 45.99%


---
## 6. GOLD - Modelisation Dimensionnelle

In [15]:
# CELLULE 15 - dim_time
df_dates = df_silver.withColumn("snapshot_date",
    F.when(F.col("last_modified_datetime").isNotNull(),
           F.to_date(F.col("last_modified_datetime")))
    .otherwise(F.current_date()))

df_dim_time = df_dates.select("snapshot_date").distinct() \
    .filter(F.col("snapshot_date").isNotNull()) \
    .withColumn("time_sk", F.dense_rank().over(Window.orderBy("snapshot_date"))) \
    .withColumn("year", year(F.col("snapshot_date"))) \
    .withColumn("month", month(F.col("snapshot_date"))) \
    .withColumn("day", dayofmonth(F.col("snapshot_date"))) \
    .withColumn("week", weekofyear(F.col("snapshot_date"))) \
    .withColumn("iso_week", weekofyear(F.col("snapshot_date"))) \
    .select(F.col("time_sk"), F.col("snapshot_date").alias("date"),
            "year", "month", "day", "week", "iso_week")

print(f"dim_time: {df_dim_time.count()} dates distinctes")

dim_time: 1587 dates distinctes


In [16]:
# CELLULE 16 - dim_brand
df_dim_brand = df_silver \
    .select(F.explode(F.split(F.col("brands"), ",")).alias("brand_name_raw")) \
    .withColumn("brand_name", F.trim(F.col("brand_name_raw"))) \
    .filter(F.col("brand_name").isNotNull() & (F.col("brand_name") != "")) \
    .dropDuplicates(["brand_name"]) \
    .withColumn("brand_sk", F.monotonically_increasing_id()) \
    .select("brand_sk", "brand_name")

print(f"dim_brand: {df_dim_brand.count()} marques")

dim_brand: 2926 marques


In [17]:
# CELLULE 17 - dim_country
df_dim_country = df_silver \
    .select(F.explode(F.split(F.col("countries_tags"), ",")).alias("country_code_raw")) \
    .withColumn("country_code", F.trim(F.col("country_code_raw"))) \
    .filter(F.col("country_code").isNotNull() & (F.col("country_code") != "")) \
    .dropDuplicates(["country_code"]) \
    .withColumn("country_sk", F.monotonically_increasing_id()) \
    .withColumn("country_name_fr", F.regexp_replace(F.col("country_code"), "^en:", "")) \
    .select("country_sk", "country_code", "country_name_fr")

print(f"dim_country: {df_dim_country.count()} pays")

dim_country: 115 pays


In [18]:
# CELLULE 18 - dim_category
df_dim_category = df_silver \
    .select(F.posexplode(F.split(F.col("categories_tags"), ",")).alias("pos", "cat_raw")) \
    .withColumn("category_code", F.trim(F.col("cat_raw"))) \
    .filter(F.col("category_code").isNotNull() & (F.col("category_code") != "")) \
    .groupBy("category_code").agg(F.min("pos").alias("level")) \
    .withColumn("level", F.col("level") + 1) \
    .withColumn("category_sk", F.monotonically_increasing_id()) \
    .withColumn("category_name_fr", F.regexp_replace(F.col("category_code"), "^en:", "")) \
    .withColumn("parent_category_sk", F.lit(None).cast("long")) \
    .select("category_sk", "category_code", "category_name_fr", "level", "parent_category_sk")

print(f"dim_category: {df_dim_category.count()} categories")

dim_category: 1662 categories


In [19]:
# CELLULE 19 - dim_nutri
df_dim_nutri = df_silver \
    .select("nutriscore_grade", "nova_group", "ecoscore_grade") \
    .dropDuplicates() \
    .filter(F.col("nutriscore_grade").isNotNull() | 
            F.col("nova_group").isNotNull() | 
            F.col("ecoscore_grade").isNotNull()) \
    .withColumn("nutri_sk", F.monotonically_increasing_id()) \
    .select("nutri_sk", "nutriscore_grade", "nova_group", "ecoscore_grade")

print(f"dim_nutri: {df_dim_nutri.count()} combinaisons")

dim_nutri: 152 combinaisons


In [20]:
# CELLULE 20 - dim_product (SCD2)
today_str = datetime.now().strftime("%Y-%m-%d")

df_dim_product = df_silver \
    .join(df_dim_brand, df_silver["brand_primary"] == df_dim_brand["brand_name"], "left") \
    .join(df_dim_category, df_silver["primary_category_code"] == df_dim_category["category_code"], "left") \
    .select(
        df_silver["code"], df_silver["product_name"],
        df_dim_brand["brand_sk"],
        df_dim_category["category_sk"].alias("primary_category_sk"),
        df_silver["countries_multi"]
    ) \
    .dropDuplicates(["code"]) \
    .withColumn("product_sk", F.monotonically_increasing_id()) \
    .withColumn("effective_from", F.to_date(F.lit(today_str))) \
    .withColumn("effective_to", F.lit(None).cast("date")) \
    .withColumn("is_current", F.lit(True)) \
    .select("product_sk", "code", "product_name", "brand_sk", "primary_category_sk",
            "countries_multi", "effective_from", "effective_to", "is_current")

print(f"dim_product: {df_dim_product.count()} produits")

dim_product: 5112 produits


In [21]:
# CELLULE 21 - bridge_product_category
df_bridge = df_silver \
    .select(F.col("code"), F.explode(F.split(F.col("categories_tags"), ",")).alias("cat_raw")) \
    .withColumn("category_code", F.trim(F.col("cat_raw"))) \
    .filter(F.col("category_code").isNotNull() & (F.col("category_code") != "")) \
    .join(df_dim_product.select("product_sk", "code"), on="code", how="inner") \
    .join(df_dim_category.select("category_sk", "category_code"), on="category_code", how="inner") \
    .select("product_sk", "category_sk") \
    .dropDuplicates()

print(f"bridge_product_category: {df_bridge.count()} relations")

bridge_product_category: 12194 relations


In [22]:
# CELLULE 22 - fact_nutrition_snapshot
df_fact_prep = df_silver.withColumn("snapshot_date",
    F.when(F.col("last_modified_datetime").isNotNull(),
           F.to_date(F.col("last_modified_datetime")))
    .otherwise(F.current_date()))

df_fact = df_fact_prep \
    .join(df_dim_product.select("product_sk", "code"), on="code", how="inner") \
    .join(df_dim_time.select(F.col("time_sk"), F.col("date").alias("snapshot_date")), 
          on="snapshot_date", how="left") \
    .withColumn("time_sk", F.coalesce(F.col("time_sk"), F.lit(1))) \
    .withColumn("fact_id", F.monotonically_increasing_id()) \
    .select(
        "fact_id", "product_sk", "time_sk",
        "energy_kcal_100g", "fat_100g", "saturated_fat_100g", "sugars_100g",
        "salt_100g", "proteins_100g", "fiber_100g", "sodium_100g",
        "nutriscore_grade", "nova_group", "ecoscore_grade",
        "completeness_score", "quality_issues_json"
    )

print(f"fact_nutrition_snapshot: {df_fact.count()} faits")

fact_nutrition_snapshot: 5112 faits


---
## 7. Chargement PostgreSQL

In [23]:
# CELLULE 23 - Chargement PostgreSQL
print("=" * 50)
print("CHARGEMENT POSTGRESQL")
print("=" * 50)

counts = {}

print("\nDimensions:")
counts["dim_time"] = load_to_pg(df_dim_time, "dim_time")
counts["dim_brand"] = load_to_pg(df_dim_brand, "dim_brand")
counts["dim_country"] = load_to_pg(df_dim_country, "dim_country")
counts["dim_category"] = load_to_pg(df_dim_category, "dim_category")
counts["dim_nutri"] = load_to_pg(df_dim_nutri, "dim_nutri")

print("\nProduits:")
counts["dim_product"] = load_to_pg(df_dim_product, "dim_product")

print("\nBridge:")
counts["bridge"] = load_to_pg(df_bridge, "bridge_product_category")

print("\nFaits:")
counts["fact"] = load_to_pg(df_fact, "fact_nutrition_snapshot")

print("\n" + "=" * 50)
print("CHARGEMENT TERMINE")
print("=" * 50)

CHARGEMENT POSTGRESQL

Dimensions:
   dim_time: 1587 lignes
   dim_brand: 2926 lignes
   dim_country: 115 lignes
   dim_category: 1662 lignes
   dim_nutri: 152 lignes

Produits:
   dim_product: 5112 lignes

Bridge:
   bridge_product_category: 12194 lignes

Faits:
   fact_nutrition_snapshot: 5112 lignes

CHARGEMENT TERMINE


---
## 8. COMPARATIF BEFORE/AFTER - Cahier de qualite

Synthese des ameliorations de qualite apres nettoyage.

In [24]:
# CELLULE 24 - Comparatif BEFORE / AFTER
print("=" * 75)
print("        CAHIER DE QUALITE - COMPARATIF BEFORE vs AFTER")
print("=" * 75)

print(f"\n{'Indicateur':<35} {'BEFORE':>12} {'AFTER':>12} {'DELTA':>12}")
print("-" * 75)

# Nombre de lignes
delta_rows = after_count - before_count
print(f"{'Nombre de lignes':<35} {before_count:>12} {after_count:>12} {delta_rows:>+12}")

# Completude globale
delta_global = avg_global_after - avg_global_before
print(f"{'Completude globale (%)':<35} {avg_global_before:>11.1f}% {avg_global_after:>11.1f}% {delta_global:>+11.1f}%")

# Completude nutriments
delta_nutri = avg_nutrient_after - avg_nutrient_before
print(f"{'Completude nutriments (%)':<35} {avg_nutrient_before:>11.1f}% {avg_nutrient_after:>11.1f}% {delta_nutri:>+11.1f}%")

print("\n" + "-" * 75)
print("COMPLETUDE PAR CHAMP:")
print("-" * 75)

all_cols = sorted(set(completude_before.keys()) | set(completude_after.keys()))
for col in all_cols:
    before_val = completude_before.get(col, 0)
    after_val = completude_after.get(col, 0)
    delta = after_val - before_val
    trend = "  +" if delta > 0 else "  -" if delta < 0 else "  ="
    print(f"  {col:<33} {before_val:>10.1f}% {after_val:>10.1f}% {delta:>+10.1f}%{trend}")

print("\n" + "-" * 75)
print("ANOMALIES CORRIGEES:")
print("-" * 75)
print(f"  Codes vides supprimes:           {anomalies_before.get('codes_vides', 0)}")
print(f"  Noms vides supprimes:            {anomalies_before.get('noms_vides', 0)}")
print(f"  Doublons elimines:               {anomalies_before.get('doublons_code', 0)}")
print(f"  Valeurs hors bornes (-> NULL):   {anomalies_before.get('hors_bornes', 0)}")
print(f"  Incoherences sat>fat (-> NULL):  {anomalies_before.get('incoherence_fat', 0)}")
print(f"\n  >>> TOTAL ANOMALIES TRAITEES:    {sum(anomalies_before.values())}")

        CAHIER DE QUALITE - COMPARATIF BEFORE vs AFTER

Indicateur                                BEFORE        AFTER        DELTA
---------------------------------------------------------------------------
Nombre de lignes                           10000         5112        -4888
Completude globale (%)                     64.7%        70.6%        +5.9%
Completude nutriments (%)                  64.3%        88.0%       +23.7%

---------------------------------------------------------------------------
COMPLETUDE PAR CHAMP:
---------------------------------------------------------------------------
  brands                                  61.4%       65.8%       +4.4%  +
  categories_tags                         42.7%       46.9%       +4.2%  +
  ecoscore_grade                          69.6%       16.1%      -53.5%  -
  energy_kcal_100g                        71.6%       98.0%      +26.4%  +
  fat_100g                                72.1%       99.4%      +27.3%  +
  fiber_100g      

In [25]:
# CELLULE 25 - Metriques JSON
metrics_json = {
    "run_timestamp": datetime.now().isoformat(),
    "source_file": INPUT_PATH,
    "pipeline": {
        "bronze_count": before_count,
        "silver_count": after_count,
        "retention_rate": round(after_count / before_count * 100, 2)
    },
    "quality_before": {
        "total_rows": before_count,
        "avg_global_completeness": avg_global_before,
        "avg_nutrient_completeness": avg_nutrient_before,
        "completeness_by_field": completude_before,
        "anomalies": anomalies_before
    },
    "quality_after": {
        "total_rows": after_count,
        "avg_global_completeness": avg_global_after,
        "avg_nutrient_completeness": avg_nutrient_after,
        "completeness_by_field": completude_after,
        "pct_nutriscore": round(df_fact.filter(F.col("nutriscore_grade").isNotNull()).count() / df_fact.count() * 100, 2),
        "pct_nova": round(df_fact.filter(F.col("nova_group").isNotNull()).count() / df_fact.count() * 100, 2)
    },
    "improvement": {
        "completeness_delta_global": round(delta_global, 2),
        "completeness_delta_nutrient": round(delta_nutri, 2),
        "anomalies_fixed": sum(anomalies_before.values())
    },
    "datamart": counts
}

print("=" * 60)
print("METRIQUES JSON (rapport complet)")
print("=" * 60)
print(json.dumps(metrics_json, indent=2, ensure_ascii=False))

METRIQUES JSON (rapport complet)
{
  "run_timestamp": "2026-01-28T19:47:32.393304",
  "source_file": "data/bronze/en.openfoodfacts.org.products_echantillon_10000.csv",
  "pipeline": {
    "bronze_count": 10000,
    "silver_count": 5112,
    "retention_rate": 51.12
  },
  "quality_before": {
    "total_rows": 10000,
    "avg_global_completeness": 64.7,
    "avg_nutrient_completeness": 64.3,
    "completeness_by_field": {
      "product_name": 92.4,
      "brands": 61.4,
      "categories_tags": 42.7,
      "nutriscore_grade": 98.9,
      "ecoscore_grade": 69.6,
      "nova_group": 25.7,
      "energy_kcal_100g": 71.6,
      "fat_100g": 72.1,
      "saturated_fat_100g": 67.2,
      "sugars_100g": 68.5,
      "salt_100g": 62.5,
      "proteins_100g": 72.3,
      "fiber_100g": 35.9
    },
    "anomalies": {
      "codes_vides": 0,
      "noms_vides": 763,
      "doublons_code": 2799,
      "hors_bornes": 93,
      "incoherence_fat": 11
    }
  },
  "quality_after": {
    "total_rows": 5112

In [26]:
# CELLULE 26 - Resume Final
print("\n" + "=" * 75)
print("                    ETL OPENFOODFACTS - RESUME FINAL")
print("=" * 75)

print(f"\n  PIPELINE:")
print(f"    Bronze (brut):     {before_count:>8} lignes")
print(f"    Silver (nettoye):  {after_count:>8} lignes")
print(f"    Retention:         {after_count/before_count*100:>7.1f}%")

print(f"\n  QUALITE (BEFORE -> AFTER):")
print(f"    Completude globale:    {avg_global_before:>5.1f}% -> {avg_global_after:>5.1f}%  ({delta_global:+.1f}%)")
print(f"    Completude nutriments: {avg_nutrient_before:>5.1f}% -> {avg_nutrient_after:>5.1f}%  ({delta_nutri:+.1f}%)")
print(f"    Anomalies corrigees:   {sum(anomalies_before.values())}")

print(f"\n  DATAMART:")
print(f"    dim_time:     {counts.get('dim_time', 0):>8} dates")
print(f"    dim_brand:    {counts.get('dim_brand', 0):>8} marques")
print(f"    dim_country:  {counts.get('dim_country', 0):>8} pays")
print(f"    dim_category: {counts.get('dim_category', 0):>8} categories")
print(f"    dim_nutri:    {counts.get('dim_nutri', 0):>8} scores")
print(f"    dim_product:  {counts.get('dim_product', 0):>8} produits")
print(f"    bridge:       {counts.get('bridge', 0):>8} relations")
print(f"    fact:         {counts.get('fact', 0):>8} faits")

print(f"\n  ETL TERMINE AVEC SUCCES")
print("=" * 75)


                    ETL OPENFOODFACTS - RESUME FINAL

  PIPELINE:
    Bronze (brut):        10000 lignes
    Silver (nettoye):      5112 lignes
    Retention:            51.1%

  QUALITE (BEFORE -> AFTER):
    Completude globale:     64.7% ->  70.6%  (+5.9%)
    Completude nutriments:  64.3% ->  88.0%  (+23.7%)
    Anomalies corrigees:   3666

  DATAMART:
    dim_time:         1587 dates
    dim_brand:        2926 marques
    dim_country:       115 pays
    dim_category:     1662 categories
    dim_nutri:         152 scores
    dim_product:      5112 produits
    bridge:          12194 relations
    fact:             5112 faits

  ETL TERMINE AVEC SUCCES


In [27]:
# CELLULE 27 - Fermer Spark
spark.stop()
print("Spark ferme")

Spark ferme
